# Bayesian Optimization Tutorial: SSN Model Recovery

This tutorial walks through using `bayesopt.py` to recover the synaptic parameters of a Stabilized Supralinear Network (SSN) from simulated data. This is a **model recovery** experiment: we first generate synthetic "target data" from a network with known parameters, then ask whether Bayesian Optimization can identify those parameters starting from a randomly initialized network.

## Prerequisites

Before running this notebook, generate the input stimuli and target data by running the two supporting scripts from the `claude_tutorial/` directory:

```bash
cd ssn-v1/claude_tutorial
python make_inputs.py
python make_targets.py
```

## Network

We use an **EI512** ring network: 256 excitatory (E) and 256 inhibitory (I) neurons arranged on a 16×16 spatial grid with a random orientation preference map (similar to a cortical column). Neurons are connected by distance- and orientation-tuned recurrent synapses.

## Stimulus

We drive the network with a set of **full-field plaid** stimuli — two orthogonal gratings (0° and 90°) at varying contrasts. Six conditions are used, spanning a range of target and mask contrast combinations.

## Optimization goal

We vary four synaptic weight parameters:

| Parameter | Meaning | True value |
|-----------|---------|------------|
| `jEE` | E→E weight | 3.839 |
| `jEI` | I→E weight (inhibitory) | −2.093 |
| `jIE` | E→I weight | 3.678 |
| `jII` | I→I weight (inhibitory) | −1.444 |

The optimizer searches for the combination that minimizes the KL divergence between simulated and target firing rate distributions.

## Algorithm overview

1. **Initialization** — Randomly sample `n_init = 50` parameter sets and evaluate each.
2. **Fit surrogate models** — Fit Gaussian Processes (GPs) to the observed costs and feasibility values.
3. **Acquisition** — Choose the next parameter set by maximizing expected improvement weighted by feasibility probability.
4. **Iterate** — Repeat steps 2–3 for `n_iter = 100` iterations.
5. **Return** — Report the parameter set with the lowest observed cost.

For a deeper introduction to Bayesian Optimization see `bayesopt_demo/bayesian_optimization_demo.ipynb`.

## 1. Imports

In [ ]:
import os
import sys
import json
import glob
import numpy as np
import h5py
import matplotlib.pyplot as plt
from itertools import combinations
from datetime import datetime
from joblib import dump, load
from sklearn.gaussian_process.kernels import Matern, WhiteKernel

# Make sure we can import ssn_v1 whether we run from claude_tutorial/ or the repo root
mainDir = os.path.abspath('.')
SSNDir  = os.path.abspath(os.path.join(mainDir, '..'))
sys.path.insert(0, mainDir)
sys.path.append(SSNDir)

from ssn_v1.SSN import SSN, NumericalInstabilityError
import ssn_v1.SSN_utils as SSN_utils
from ssn_v1.bayesopt import bayesopt

print(f'Working directory: {mainDir}')

## 2. Initialize the Network

We load the network configuration from `config.EI512_tutorial.json`. This file points to:
- `network/config.network.EI512_tutorial.json` — architecture (ring network, 256 E + 256 I cells)
- `components/SSN_cells/` — cell-type parameters (τ, k, n, c)

The network is a **random ring** with a smooth orientation preference map (generated by Kaschube's planform method). Nodes are arranged on a 16×16 spatial grid. Recurrent connections are tuned in both space and orientation.

Here we initialize the network and build it once to inspect its structure. During optimization we will rebuild it many times with different parameter values.

In [ ]:
config_file = os.path.join(mainDir, 'config.EI512_tutorial.json')

V1net = SSN('EI512_tutorial')
V1net.load_config(config_file)
print(f'Loaded config: {config_file}')

# Build the network with fixed seeds so the result is reproducible
V1net.set_rand_seed(seed=42)
V1net.add_nodes()
V1net.add_edges(seed=42)

is_E = V1net.nodes['ei'].values == 'e'
is_I = V1net.nodes['ei'].values == 'i'
print(f'N_E = {is_E.sum()},  N_I = {is_I.sum()}')

In [ ]:
# Quick look at the spatial layout and orientation preferences
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

Ex  = V1net.nodes.loc[is_E, 'x']
Ey  = V1net.nodes.loc[is_E, 'y']
Ix  = V1net.nodes.loc[is_I, 'x']
Iy  = V1net.nodes.loc[is_I, 'y']
Eori = V1net.nodes.loc[is_E, 'orientation']
Iori = V1net.nodes.loc[is_I, 'orientation']

sc = axes[0].scatter(Ex, Ey, c=Eori, s=12, marker='^', cmap='hsv', label='E')
axes[0].scatter(Ix, Iy, c=Iori, s=12, marker='s', cmap='hsv', label='I')
fig.colorbar(sc, ax=axes[0], label='Orientation pref. (rad)')
axes[0].set_title('Node layout and orientation preferences')
axes[0].set_xlabel('x (deg)')
axes[0].set_ylabel('y (deg)')
axes[0].legend()

axes[1].hist(np.degrees(V1net.nodes.loc[is_E, 'orientation']),
             bins=30, alpha=0.6, label='E', color='royalblue')
axes[1].hist(np.degrees(V1net.nodes.loc[is_I, 'orientation']),
             bins=30, alpha=0.6, label='I', color='tomato')
axes[1].set_title('Distribution of orientation preferences')
axes[1].set_xlabel('Preferred orientation (deg)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Load Inputs and Targets

### Inputs

Each input file (generated by `make_inputs.py`) is a plaid stimulus array of shape **(NX, NY, NT, NOri)** = (16, 16, 60, 16). It encodes, for each spatial grid point, how strongly each of the 16 orientation channels (0–168.75°) is driven at each time step. The temporal envelope is a transient-sustained waveform (onset burst + plateau).

When `SSN.connect_inputs()` is called, the SSN maps this spatial array to per-neuron input currents **(N_cells, NT)** based on each neuron's spatial position and orientation preference.

### Targets

Each target file (generated by `make_targets.py`) stores the **final-time-step firing rates** from the true network: a vector of shape **(N_cells,)**, one rate per neuron. These are the distributions we want the model to match.

In [ ]:
# ---------------------------------------------------------------------------- #
#  Load inputs                                                                  #
# ---------------------------------------------------------------------------- #
input_files  = sorted(glob.glob(os.path.join(mainDir, 'inputs', 'plaid_*.h5')))
target_files = sorted(glob.glob(os.path.join(mainDir, 'targets', 'target_*.h5')))

if not input_files:
    raise FileNotFoundError('No input files found. Run make_inputs.py first.')
if not target_files:
    raise FileNotFoundError('No target files found. Run make_targets.py first.')

# input_data maps condition label -> (NX, NY, NT, NOri) array
input_data = {}
for fpath in input_files:
    key = os.path.splitext(os.path.basename(fpath))[0]  # e.g. 'plaid_C0.00_maskC0.05'
    with h5py.File(fpath, 'r') as hf:
        input_data[key] = hf['stimulus'][:]

# ground_truth_data: (N_cells, N_cond) matrix of final firing rates
N_cells = len(V1net.nodes)
N_cond  = len(target_files)
ground_truth_data = np.zeros((N_cells, N_cond))

for c_i, fpath in enumerate(target_files):
    with h5py.File(fpath, 'r') as hf:
        ground_truth_data[:, c_i] = hf['rates'][:]

print(f'Loaded {len(input_data)} stimulus conditions')
print(f'Input array shape (one condition): {next(iter(input_data.values())).shape}')
print(f'Ground truth data shape: {ground_truth_data.shape}')

In [ ]:
# Visualize: histogram of target E and I firing rates across all conditions
target_E = ground_truth_data[is_E, :].ravel()
target_I = ground_truth_data[is_I, :].ravel()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].hist(target_E, bins=40, color='royalblue', edgecolor='none')
axes[0].set_title('Target E-cell firing rates (all conditions)')
axes[0].set_xlabel('Firing rate (spk/s)')
axes[0].set_ylabel('Count')

axes[1].hist(target_I, bins=40, color='tomato', edgecolor='none')
axes[1].set_title('Target I-cell firing rates (all conditions)')
axes[1].set_xlabel('Firing rate (spk/s)')

plt.tight_layout()
plt.show()

print(f'E rates: mean={target_E.mean():.2f}  max={target_E.max():.2f} spk/s')
print(f'I rates: mean={target_I.mean():.2f}  max={target_I.max():.2f} spk/s')

In [ ]:
# Visualize: histogram of input values (final time step, one example condition)
example_key  = sorted(input_data.keys())[5]  # C=0.50, maskC=0.50
example_stim = input_data[example_key]        # (NX, NY, NT, NOri)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].hist(example_stim[:, :, -1, :].ravel(), bins=40, color='slategray', edgecolor='none')
axes[0].set_title(f'Input values at final time step\n({example_key})')
axes[0].set_xlabel('Input amplitude')
axes[0].set_ylabel('Count')

# Temporal profile at a sample spatial position
t_profile = example_stim[8, 8, :, :].mean(axis=-1)  # mean across ori channels, centre pixel
t_axis = np.linspace(0, 3000, len(t_profile))

axes[1].plot(t_axis, t_profile, color='slategray')
axes[1].set_title('Temporal envelope (centre pixel, mean over ori channels)')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Input amplitude')

plt.tight_layout()
plt.show()

## 4. Parameter Evaluation Functions

Bayesian Optimization requires a function `evaluate_parameters` that, given a candidate parameter set, returns a **(feasibility, cost)** pair.

### Helper functions

We first define two helper functions:

- **`build_network`** — Constructs an SSN from the config file, then injects a new set of j values into the parameters dict before calling `add_nodes()` and `add_edges()`.
- **`run_simulation`** — Assigns the raw stimulus array to the network, calls `connect_inputs()` to map spatial inputs to per-neuron currents, then runs the ODE solver.
- **`feasibility_check`** — Returns 1 if all firing rates stay below a threshold, 0 otherwise. Explosively high firing rates indicate an unstable (infeasible) parameter set.

In [ ]:
def build_network(config_file, theta, param_map, seed=0, fix_nodes=False, node_seed=0):
    """Build an SSN with parameter values taken from `theta`."""
    ssn = SSN('ei_model', verbose=False)
    ssn.load_config(config_file)

    # Set the random seed that determines the orientation map and node layout
    if fix_nodes:
        ssn.set_rand_seed(seed=node_seed)
    else:
        ssn.set_rand_seed(seed=seed)

    # Inject candidate parameter values
    for i, (name, path) in enumerate(param_map.items()):
        d = ssn.parameters
        for key in path[:-1]:
            d = d[key]
        d[path[-1]] = theta[i]

    ssn.add_nodes()
    ssn.add_edges(seed=seed)
    return ssn


def run_simulation(ssn, input_matrix):
    """Map `input_matrix` (NX, NY, NT, NOri) to per-neuron currents and run."""
    ssn.inputs = input_matrix
    ssn.connect_inputs()
    ssn.run()
    return ssn


def feasibility_check(ssn, threshold=1e5):
    """Return 1 if firing rates are bounded, 0 if they exceeded `threshold`."""
    if (ssn.outputs.y > threshold).any():
        return 0
    return 1

### Cost function: per-cell-type KL divergence

We measure how well model firing rates match the targets using the **Kullback–Leibler divergence**:

$$D_{KL}(P \| Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}$$

where $P$ is the target distribution and $Q$ is the model distribution.  Lower KL means the distributions are more similar.

Rather than binning the data into histograms (which requires choosing bin edges), we use the **Wang–Kulkarni–Verdú (WKV) k-nearest-neighbors estimator** (Wang et al. 2009), which works directly on samples:

$$\hat{D}_{KL}(P \| Q) \approx \frac{d}{n} \sum_{i=1}^{n} \log \frac{\nu_i}{\rho_i} + \log \frac{m}{n-1}$$

where $\rho_i$ is the distance from sample $x_i$ to its $k$-th nearest neighbor *within* $P$, and $\nu_i$ is the distance from $x_i$ to its $k$-th nearest neighbor *in* $Q$.

We compute KL **separately for E and I cells**, then sum:

$$\text{cost} = D_{KL}^{(E)} + D_{KL}^{(I)}$$

This is done per stimulus condition and then accumulated across all conditions.

In [ ]:
def evaluate_parameters(
    theta,
    param_map,
    ground_truth_data,
    input_data,
    n_inst=1,
    seed=0,
    fix_seed=False,
    fix_nodes=False,
    node_seed=0,
    **kwargs,
):
    """
    Evaluate a candidate parameter set.

    Parameters
    ----------
    theta : np.ndarray, shape (n_params,)
        Candidate parameter values [jEE, jEI, jIE, jII].
    param_map : dict
        Maps parameter names to paths in the SSN config.
    ground_truth_data : np.ndarray, shape (N_cells, N_cond)
        Target firing rates from the true network.
    input_data : dict
        Maps condition labels to raw stimulus arrays (NX, NY, NT, NOri).

    Returns
    -------
    feas_value : float
        Fraction of n_inst runs that were stable (0 or 1 when n_inst=1).
    cost_value : float
        Sum of per-cell-type WKV KL divergences; 1e9 if all runs were infeasible.
    """
    cfg_file  = kwargs.get('config_file', os.path.join(mainDir, 'config.EI512_tutorial.json'))
    threshold = kwargs.get('feas_threshold', 1e5)
    k_knn     = kwargs.get('k_knn', 5)

    rng         = np.random.default_rng(seed)
    stable_count = 0
    accum_cost   = 0.0

    cond_keys = list(input_data.keys())
    N_cond    = len(cond_keys)

    for _ in range(n_inst):
        subseed = seed if fix_seed else int(rng.integers(0, 9_999_999))

        model = build_network(cfg_file, theta, param_map,
                              seed=subseed, fix_nodes=fix_nodes, node_seed=node_seed)

        # Cell-type masks for this model instance
        is_E_m = model.nodes['ei'].values == 'e'
        is_I_m = model.nodes['ei'].values == 'i'

        N_cells   = len(model.nodes)
        sim_rates = np.zeros((N_cells, N_cond))

        feas = 1
        for c_i, key in enumerate(cond_keys):
            try:
                model = run_simulation(model, input_data[key])
            except NumericalInstabilityError:
                feas = 0
                break
            feas = feasibility_check(model, threshold=threshold)
            if feas == 0:
                break
            sim_rates[:, c_i] = model.outputs.y[:, -1]

        if feas == 1:
            stable_count += 1

            # KL divergence per cell type, per condition — summed across both
            kl_cost = 0.0
            for c_i in range(N_cond):
                kl_cost += SSN_utils.kl_divergence(
                    sim_rates[is_E_m, c_i],
                    ground_truth_data[is_E_m, c_i],
                    method='wkv_knn', k=k_knn,
                )
                kl_cost += SSN_utils.kl_divergence(
                    sim_rates[is_I_m, c_i],
                    ground_truth_data[is_I_m, c_i],
                    method='wkv_knn', k=k_knn,
                )
            accum_cost += kl_cost

    feas_value = stable_count / n_inst
    cost_value = accum_cost / stable_count if stable_count > 0 else 1e9

    return feas_value, cost_value

## 5. Run Bayesian Optimization

### Hyperparameters

| Setting | Value | Rationale |
|---------|-------|-----------|
| `n_init` | 50 | Random samples before fitting any GP |
| `n_iter` | 100 | GP-guided iterations after initialization |
| `n_inst` | 1 | One random network instantiation per evaluation |
| `random_state` | 42 | Reproducibility |
| `fix_nodes` | True | Fix node layout (same orientation map every run) |
| `node_seed` | 42 | Seed for the fixed node layout |

### Kernel fit schedule

Fitting GP kernel hyperparameters at every iteration is expensive. We use a schedule:
- **Iterations 0–49**: fit every iteration (the GP has few data points and benefits from frequent refit)
- **Iterations 50–99**: fit every 5 iterations (the GP is more stable with more data)

### Parameter bounds

The bounds are set wide enough to contain the true values with room to explore.

In [ ]:
# True parameter values (what we hope to recover)
true_params = {'jEE': 3.839, 'jEI': -2.093, 'jIE': 3.678, 'jII': -1.444}

# Search bounds — each tuple is (lower, upper)
param_bounds = {
    'jEE': [0.5,  8.0],
    'jEI': [-4.5, -0.5],
    'jIE': [0.5,  8.0],
    'jII': [-3.5, -0.3],
}

# Map from parameter name to its location in the SSN parameters dict
param_map = {
    'jEE': ('edges', 'spatial_tuning', 'E<-E', 'j'),
    'jEI': ('edges', 'spatial_tuning', 'E<-I', 'j'),
    'jIE': ('edges', 'spatial_tuning', 'I<-E', 'j'),
    'jII': ('edges', 'spatial_tuning', 'I<-I', 'j'),
}

# Keyword arguments forwarded to evaluate_parameters
evaluation_kwargs = {
    'config_file':    os.path.join(mainDir, 'config.EI512_tutorial.json'),
    'feas_threshold': 1e5,
    'k_knn':          5,
}

# Kernel-fit schedule: fit every iteration for the first 50, then every 5
n_init = 50
n_iter = 100
kernel_fit_schedule = list(range(n_init)) + list(range(n_init, n_iter, 5))

print('Parameter bounds:')
for k, (lo, hi) in param_bounds.items():
    tv = true_params[k]
    print(f'  {k:5s}  [{lo:5.1f}, {hi:5.1f}]   true = {tv}')

In [ ]:
# GP kernels — Matern + WhiteKernel so the noise level is fitted adaptively
# rather than fixed. This prevents the Cholesky decomposition from failing
# when many feasibility values are identical or cost values span a wide range.
n_params = len(param_bounds)
kernel_cost = (Matern(nu=2.5, length_scale=np.ones(n_params), length_scale_bounds=(0.001, 100.0))
               + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-8, 1e2)))
kernel_feas = (Matern(nu=2.5, length_scale=np.ones(n_params), length_scale_bounds=(0.001, 100.0))
               + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-8, 1e2)))

# Create the bayesopt object
bo = bayesopt(
    param_bounds,
    param_map,
    user_evaluate=evaluate_parameters,
    evaluation_kwargs=evaluation_kwargs,
)

# Run the optimization
best_params = bo.bayesopt(
    n_init=n_init,
    n_inst=1,
    n_iter=n_iter,
    random_state=42,
    acquisition_func=SSN_utils.expected_improvement,
    ground_truth_data=ground_truth_data,
    input_data=input_data,
    fix_seed=True,
    fix_nodes=True,
    node_seed=42,
    use_feas=True,
    kernel_cost=kernel_cost,
    kernel_feas=kernel_feas,
    kernel_fit_schedule=kernel_fit_schedule,
    verbose=False,
)

print('\nBest parameters found:')
for k, v in best_params.items():
    tv = true_params[k]
    print(f'  {k:5s}  found={v:7.4f}   true={tv:7.4f}   err={abs(v-tv):.4f}')

### Save the training run

We save the `bayesopt` object to disk. It contains the full optimization history (`history_params`, `history_costs`, `history_feas`) and the fitted GP models (`gp_cost`, `gp_feas`).

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
save_path = os.path.join(mainDir, f'bo_run_{timestamp}.joblib')
dump(bo, save_path)
print(f'Saved: {save_path}')

## 6. Visualize Results

### 6.1 Cost over time

`bo.history_minCosts` stores the best (lowest) cost seen up to each iteration. We expect it to decrease as the optimizer finds better parameter sets.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(bo.history_minCosts, color='steelblue')
ax.set_xlabel('Iteration')
ax.set_ylabel('Minimum KL cost (E + I, all conditions)')
ax.set_title('Cost history')
ax.axvline(n_init, color='gray', linestyle='--', label=f'End of initialization (n={n_init})')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(mainDir, 'plots', 'cost_history.png'), dpi=150)
plt.show()

### 6.2 GP slices — cost landscape

`bo.plot_gp_slice_2d` shows 2D slices of the GP-estimated cost landscape centred on the discovered optimum. All other parameters are held fixed at the best-found values. Left panel: predicted mean cost. Right panel: predicted standard deviation (uncertainty).

The red star marks the true parameter values. The blue dot marks the GP's estimated minimum.

In [ ]:
param_pairs = list(combinations(list(param_bounds.keys()), 2))

for (p1, p2) in param_pairs:
    fig, ax = bo.plot_gp_slice_2d([p1, p2])
    ax[0].scatter(true_params[p1], true_params[p2],
                  c='r', marker='*', s=150, zorder=5, label='True params')
    ax[0].legend()
    fig.suptitle(f'Cost GP — {p1} vs {p2}', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(mainDir, 'plots', f'cost_gp_{p1}_vs_{p2}.png'), dpi=150)
    plt.show()

### 6.3 GP slices — feasibility landscape

We can show the same slices for the feasibility GP. High feasibility (bright) means that parameter set tends to produce stable simulations. The optimizer naturally avoids low-feasibility regions.

In [ ]:
for (p1, p2) in param_pairs:
    fig, ax = bo.plot_gp_slice_2d([p1, p2], gp=bo.gp_feas)
    ax[0].scatter(true_params[p1], true_params[p2],
                  c='r', marker='*', s=150, zorder=5, label='True params')
    ax[0].legend()
    fig.suptitle(f'Feasibility GP — {p1} vs {p2}', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(mainDir, 'plots', f'feas_gp_{p1}_vs_{p2}.png'), dpi=150)
    plt.show()

### 6.4 Parameter trace: convergence toward true values

We plot all sampled parameter sets as scatter plots in 2D projections of the parameter space. Points are coloured by iteration (dark = early, bright = late). The red star marks the true parameters.

We expect the cloud of sampled points to tighten around the true values as training progresses.

In [ ]:
def plot_param_slices(history_params, true_params_list, param_names,
                      cmap='Blues', marker_size=8):
    """Scatter plot of sampled parameters across 2D projections, coloured by iteration."""
    history_params  = np.array(history_params)
    true_params_arr = np.array(true_params_list)
    n_params = history_params.shape[1]
    n_pairs  = n_params * (n_params - 1) // 2

    fig, axes = plt.subplots(n_params - 1, n_params - 1,
                             figsize=(4 * (n_params - 1), 4 * (n_params - 1)))
    fig.subplots_adjust(right=0.9)

    for i, j in combinations(range(n_params), 2):
        ax = axes[i, j - 1] if n_params > 2 else axes

        sc = ax.scatter(
            history_params[:, i], history_params[:, j],
            c=range(len(history_params)), s=marker_size,
            cmap=cmap, vmin=0, vmax=len(history_params) - 1,
        )
        ax.plot(true_params_arr[i], true_params_arr[j],
                'r*', markersize=15, label='True')
        ax.set_xlabel(param_names[i])
        ax.set_ylabel(param_names[j])
        ax.set_title(f'{param_names[i]} vs {param_names[j]}')
        if i == 0 and j == 1:
            ax.legend()

    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(sc, cax=cbar_ax)
    cbar.set_label('Iteration')

    plt.tight_layout(rect=[0, 0, 0.9, 1])
    return fig, axes


param_names = list(true_params.keys())
true_vals   = [true_params[k] for k in param_names]

fig, ax = plot_param_slices(
    bo.history_params, true_vals, param_names, cmap='Blues'
)
plt.savefig(os.path.join(mainDir, 'plots', 'param_trace.png'), dpi=150)
plt.show()

### 6.5 Parameter trace: feasibility

The same scatter but coloured by feasibility value (0 = infeasible, 1 = stable). This shows which regions of parameter space are physically plausible.

In [ ]:
def plot_feasibility_slices(history_params, history_feas, true_params_list,
                            param_names, cmap='copper', marker_size=8):
    """Scatter plot of sampled parameters coloured by feasibility."""
    history_params  = np.array(history_params)
    history_feas    = np.array(history_feas)
    true_params_arr = np.array(true_params_list)
    n_params = history_params.shape[1]

    fig, axes = plt.subplots(n_params - 1, n_params - 1,
                             figsize=(4 * (n_params - 1), 4 * (n_params - 1)))
    fig.subplots_adjust(right=0.9)

    for i, j in combinations(range(n_params), 2):
        ax = axes[i, j - 1] if n_params > 2 else axes

        sc = ax.scatter(
            history_params[:, i], history_params[:, j],
            c=history_feas, s=marker_size, cmap=cmap, vmin=0, vmax=1,
        )
        ax.plot(true_params_arr[i], true_params_arr[j],
                'r*', markersize=15, label='True')
        ax.set_xlabel(param_names[i])
        ax.set_ylabel(param_names[j])
        ax.set_title(f'{param_names[i]} vs {param_names[j]}')
        if i == 0 and j == 1:
            ax.legend()

    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(sc, cax=cbar_ax)
    cbar.set_label('Feasibility')

    plt.tight_layout(rect=[0, 0, 0.9, 1])
    return fig, axes


fig, ax = plot_feasibility_slices(
    bo.history_params, bo.history_feas, true_vals, param_names, cmap='copper'
)
plt.savefig(os.path.join(mainDir, 'plots', 'feas_trace.png'), dpi=150)
plt.show()

### 6.6 Distance to true parameters over time

A simple summary metric is the **Euclidean distance** between each sampled parameter set and the ground truth. If the optimizer is converging on the correct solution, this distance should trend downward. We plot it alongside the minimum cost.

In [ ]:
history_params  = np.array(bo.history_params)
history_minCosts = np.array(bo.history_minCosts)
true_arr         = np.array(true_vals)

# Euclidean distance from each sampled point to the true parameters
distances = np.linalg.norm(history_params - true_arr, axis=1)

fig, ax1 = plt.subplots(figsize=(10, 4))

color1 = 'steelblue'
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Euclidean distance to true params', color=color1)
ax1.plot(distances, color=color1, alpha=0.7, label='Distance')
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = 'tomato'
ax2.set_ylabel('Min cost so far', color=color2)
ax2.plot(history_minCosts, color=color2, linestyle='--', label='Min cost')
ax2.tick_params(axis='y', labelcolor=color2)

ax1.axvline(n_init, color='gray', linestyle=':', label=f'n_init={n_init}')
ax1.legend(loc='upper right')
plt.title('Convergence toward true parameters')
plt.tight_layout()
plt.savefig(os.path.join(mainDir, 'plots', 'convergence.png'), dpi=150)
plt.show()